# Superstore Sales Intelligence - Executive Analysis
**Dataset:** Retail transactional data · 2015–2018  
**Scope:** Revenue trends, customer segmentation, RFM scoring, regional and fulfillment analysis  
**Goal:** Extract actionable business insights from raw transaction logs — the same workflow I apply to SaaS product analytics and e-commerce clients.

---

## 1. Setup & imports

I keep imports organized by purpose: data wrangling, visualization, and display utilities.
Setting plot style early avoids repeating it in every figure.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# Consistent visual identity across all charts
plt.rcParams.update({
    'figure.facecolor': 'white',
    'axes.facecolor':   '#f8f8f6',
    'axes.spines.top':  False,
    'axes.spines.right':False,
    'axes.grid':        True,
    'grid.color':       'white',
    'grid.linewidth':   1.0,
    'font.family':      'DejaVu Sans',
    'axes.labelcolor':  '#444',
    'xtick.color':      '#666',
    'ytick.color':      '#666',
})

# Color palette — reuse throughout to keep charts coherent
GREEN  = '#1D9E75'
BLUE   = '#378ADD'
AMBER  = '#EF9F27'
CORAL  = '#D85A30'
PURPLE = '#7F77DD'
GRAY   = '#888780'

print('Libraries loaded.')

## 2. Load & inspect the raw data

First pass: understand shape, dtypes, and obvious quality issues before touching anything.

In [ ]:
df_raw = pd.read_csv('base_teste.csv')

print(f'Shape: {df_raw.shape}')
print(f'\nColumn dtypes:')
print(df_raw.dtypes)
print(f'\nNull counts:')
print(df_raw.isnull().sum())
print(f'\nSample rows:')
df_raw.head(3)

## 3. Data cleaning & type coercion

Two issues caught during inspection:

1. **`Sales` is a string** — the source uses a mixed decimal/thousands separator format (e.g. `9.575.775` instead of `9575.775`). I'll write a parser to handle both.
2. **Dates are strings** — `Order Date` and `Ship Date` need to be datetime so I can compute lead times and time-series aggregations.

Everything else looks clean: no nulls in key columns, consistent category values.

In [ ]:
df = df_raw.copy()

# ── Fix Sales ──────────────────────────────────────────────────────────────
# The source file has dots both as thousands separators AND decimal points.
# Strategy: split on '.', if there are 3+ parts, everything except the last
# segment is integer digits; rejoin without the thousands dots.
def parse_sales(val: str) -> float:
    val = str(val).strip()
    parts = val.split('.')
    if len(parts) > 2:
        # e.g. '9.575.775'  ->  '9575.775'
        val = ''.join(parts[:-1]) + '.' + parts[-1]
    return float(val)

df['Sales'] = df['Sales'].apply(parse_sales)

# ── Parse dates ────────────────────────────────────────────────────────────
# dayfirst=True because the format is DD/MM/YYYY
df['Order Date'] = pd.to_datetime(df['Order Date'], dayfirst=True)
df['Ship Date']  = pd.to_datetime(df['Ship Date'],  dayfirst=True)

# ── Derived columns ────────────────────────────────────────────────────────
df['Year']       = df['Order Date'].dt.year
df['Month']      = df['Order Date'].dt.to_period('M')  # e.g. 2017-09
df['Quarter']    = df['Order Date'].dt.to_period('Q')  # e.g. 2017Q3
df['lead_days']  = (df['Ship Date'] - df['Order Date']).dt.days

# Quick sanity check
print('Sales dtype:', df['Sales'].dtype)
print('Date range:', df['Order Date'].min().date(), '->', df['Order Date'].max().date())
print('Lead time range:', df['lead_days'].min(), '->', df['lead_days'].max(), 'days')
print('Any negatives?', (df['lead_days'] < 0).sum())
df[['Sales','Order Date','Ship Date','lead_days']].describe()

## 4. Revenue trend analysis

Starting with the big picture: annual totals and month-over-month to spot growth trajectory and seasonality patterns.

In [ ]:
# ── Annual aggregation ─────────────────────────────────────────────────────
annual = df.groupby('Year').agg(
    revenue  = ('Sales',      'sum'),
    orders   = ('Order ID',   'nunique'),
    customers= ('Customer ID','nunique'),
).round(2)

annual['aov']     = (annual['revenue'] / annual['orders']).round(2)  # avg order value
annual['yoy_pct'] = annual['revenue'].pct_change().mul(100).round(1) # year-over-year %

print('Annual summary:')
print(annual.to_string())

# ── Monthly for trend chart ────────────────────────────────────────────────
monthly = (
    df.groupby('Month')['Sales']
    .sum()
    .reset_index()
    .rename(columns={'Sales': 'revenue'})
)
monthly['month_dt'] = monthly['Month'].dt.to_timestamp()
monthly['rolling_3m'] = monthly['revenue'].rolling(3, center=True).mean()  # smooth trend

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# Left: monthly revenue + rolling average
ax = axes[0]
ax.bar(monthly['month_dt'], monthly['revenue'], color=GREEN, alpha=0.45, width=25, label='Monthly')
ax.plot(monthly['month_dt'], monthly['rolling_3m'], color=GREEN, linewidth=2.5, label='3-month avg')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x/1000:.0f}K'))
ax.set_title('Monthly revenue with 3-month rolling average', fontsize=12, pad=10)
ax.legend(fontsize=9)

# Right: annual bars + YoY line on secondary axis
ax2 = axes[1]
bars = ax2.bar(annual.index, annual['revenue'], color=[GREEN]*4, alpha=0.75, width=0.5)
# Highlight best year
bars[annual['revenue'].argmax()].set_color(GREEN)
bars[annual['revenue'].argmax()].set_alpha(1.0)

ax2b = ax2.twinx()
yoy_vals = annual['yoy_pct'].dropna()
ax2b.plot(yoy_vals.index, yoy_vals.values, color=AMBER, marker='o', linewidth=2, markersize=6, label='YoY %')
ax2b.set_ylabel('YoY growth (%)', color=AMBER, fontsize=10)
ax2b.tick_params(axis='y', labelcolor=AMBER)
ax2b.spines['right'].set_visible(True)
ax2b.spines['right'].set_color(AMBER)
ax2.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x/1000:.0f}K'))
ax2.set_title('Annual revenue & YoY growth', fontsize=12, pad=10)

# Annotate bars with value
for bar in bars:
    ax2.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 5000,
             f'${bar.get_height()/1000:.0f}K', ha='center', va='bottom', fontsize=9, color='#444')

plt.tight_layout()
plt.savefig('fig_revenue_trend.png', dpi=150, bbox_inches='tight')
plt.show()
print('\nKey insight: Revenue grew 54% from 2015 to 2018, with the strongest jump in 2017 (+25.5%)')

## 5. Average Order Value (AOV) — a counter-intuitive signal

Revenue is growing, but the AOV is actually declining. This is an important distinction:
the business is scaling by volume, not by ticket size. Worth flagging to any stakeholder.

In [ ]:
fig, ax = plt.subplots(figsize=(7, 3.5))

ax.plot(annual.index, annual['aov'], color=CORAL, linewidth=2.5, marker='o', markersize=8)
ax.fill_between(annual.index, annual['aov'], alpha=0.12, color=CORAL)

for yr, val in annual['aov'].items():
    ax.annotate(f'${val:.0f}', (yr, val), textcoords='offset points',
                xytext=(0, 10), ha='center', fontsize=10, color=CORAL)

ax.set_ylim(400, 560)
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x:.0f}'))
ax.set_title('Average Order Value by year — declining despite revenue growth', fontsize=12, pad=10)
plt.tight_layout()
plt.savefig('fig_aov.png', dpi=150, bbox_inches='tight')
plt.show()

delta = annual['aov'].iloc[-1] - annual['aov'].iloc[0]
print(f'AOV change 2015→2018: {delta:+.2f} USD ({delta/annual["aov"].iloc[0]*100:.1f}%)')
print('Implication: the store is getting more orders, but each order is smaller.')
print('Recommendation: upsell/cross-sell flows, bundle promotions, minimum order thresholds.')

## 6. Customer segment breakdown

The dataset has three segments: Consumer, Corporate, and Home Office.
I want to understand not just revenue share, but also order volume and AOV per segment —
because a segment can look small in revenue but have high ticket size (high-value, low-frequency).

In [ ]:
seg = df.groupby('Segment').agg(
    revenue  = ('Sales',      'sum'),
    orders   = ('Order ID',   'nunique'),
    customers= ('Customer ID','nunique'),
).round(2)

seg['aov']     = (seg['revenue'] / seg['orders']).round(2)
seg['rev_pct'] = (seg['revenue'] / seg['revenue'].sum() * 100).round(1)
seg['orders_pct'] = (seg['orders'] / seg['orders'].sum() * 100).round(1)

print(seg.to_string())

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(13, 4))
colors_seg = [GREEN, BLUE, AMBER]
labels = seg.index.tolist()

# Revenue share
axes[0].pie(seg['rev_pct'], labels=labels, colors=colors_seg,
            autopct='%1.1f%%', startangle=90,
            wedgeprops={'linewidth': 2, 'edgecolor': 'white'})
axes[0].set_title('Revenue share', fontsize=11)

# Order count
axes[1].bar(labels, seg['orders'], color=colors_seg, width=0.5)
axes[1].set_title('Order count', fontsize=11)
axes[1].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x:,.0f}'))
for i, (lbl, val) in enumerate(zip(labels, seg['orders'])):
    axes[1].text(i, val + 30, f'{val:,}', ha='center', fontsize=9, color='#444')

# AOV comparison
axes[2].bar(labels, seg['aov'], color=colors_seg, width=0.5)
axes[2].set_title('Avg order value (AOV)', fontsize=11)
axes[2].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x:.0f}'))
axes[2].set_ylim(460, 520)
for i, (lbl, val) in enumerate(zip(labels, seg['aov'])):
    axes[2].text(i, val + 1, f'${val:.0f}', ha='center', fontsize=9, color='#444')

plt.suptitle('Customer segment analysis', fontsize=13, y=1.02)
plt.tight_layout()
plt.savefig('fig_segments.png', dpi=150, bbox_inches='tight')
plt.show()

## 7. Product category & sub-category analysis

In [ ]:
cat = (
    df.groupby('Category')['Sales']
    .sum()
    .sort_values(ascending=False)
    .reset_index()
)

subcat = (
    df.groupby('Sub-Category')['Sales']
    .sum()
    .sort_values(ascending=False)
    .head(10)
    .reset_index()
)

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# Category bars
colors_cat = [GREEN, BLUE, AMBER]
axes[0].barh(cat['Category'], cat['Sales'], color=colors_cat, height=0.5)
axes[0].xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x/1000:.0f}K'))
axes[0].set_title('Revenue by category', fontsize=11)
for i, val in enumerate(cat['Sales']):
    axes[0].text(val + 5000, i, f'${val/1000:.0f}K', va='center', fontsize=9, color='#444')

# Sub-category ranked bar
axes[1].barh(subcat['Sub-Category'], subcat['Sales'], color=BLUE, alpha=0.75, height=0.6)
axes[1].xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x/1000:.0f}K'))
axes[1].set_title('Top 10 sub-categories', fontsize=11)
axes[1].invert_yaxis()  # highest value at top

plt.tight_layout()
plt.savefig('fig_categories.png', dpi=150, bbox_inches='tight')
plt.show()

## 8. RFM customer segmentation

RFM (Recency, Frequency, Monetary) is the standard framework for customer value segmentation.
The idea: customers who bought recently, buy often, and spend more are more valuable.

**Implementation:** I score each customer from 1–4 on each dimension using quantile bins,
then sum to a single score (3–12) and map to human-readable segments.

Why quantile bins? They guarantee balanced segment sizes regardless of skew in the data —
which is almost always present in retail transaction data.

In [ ]:
# Reference date: one day after the last transaction
REF_DATE = df['Order Date'].max() + pd.Timedelta(days=1)

rfm = df.groupby('Customer ID').agg(
    customer_name = ('Customer Name', 'first'),
    recency_days  = ('Order Date',    lambda x: (REF_DATE - x.max()).days),
    frequency     = ('Order ID',      'nunique'),   # distinct orders
    monetary      = ('Sales',         'sum'),
    first_order   = ('Order Date',    'min'),
    last_order    = ('Order Date',    'max'),
).reset_index()

# Score R: lower recency = better = higher score (reverse)
rfm['R'] = pd.qcut(rfm['recency_days'], 4, labels=[4, 3, 2, 1]).astype(int)

# Score F and M: higher = better = higher score
# Using rank(method='first') to break ties deterministically
rfm['F'] = pd.qcut(rfm['frequency'].rank(method='first'), 4, labels=[1, 2, 3, 4]).astype(int)
rfm['M'] = pd.qcut(rfm['monetary'],                       4, labels=[1, 2, 3, 4]).astype(int)

rfm['RFM_score'] = rfm['R'] + rfm['F'] + rfm['M']  # range: 3–12

# Map scores to business segments
def rfm_label(score: int) -> str:
    if score >= 10: return 'Champions'
    if score >= 8:  return 'Loyal'
    if score >= 6:  return 'Potential'
    if score >= 4:  return 'At Risk'
    return 'Lost'

rfm['segment'] = rfm['RFM_score'].apply(rfm_label)

# Segment summary
rfm_summary = rfm.groupby('segment').agg(
    customers = ('Customer ID', 'count'),
    revenue   = ('monetary',    'sum'),
    avg_orders= ('frequency',   'mean'),
    avg_spend = ('monetary',    'mean'),
).round(2)

rfm_summary['rev_pct'] = (rfm_summary['revenue'] / rfm_summary['revenue'].sum() * 100).round(1)
print(rfm_summary.to_string())

In [ ]:
order  = ['Champions', 'Loyal', 'Potential', 'At Risk', 'Lost']
colors_rfm = [GREEN, BLUE, PURPLE, AMBER, CORAL]
rfm_plot = rfm_summary.reindex(order)

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# Revenue by segment
axes[0].bar(order, rfm_plot['revenue'], color=colors_rfm, width=0.55)
axes[0].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x/1000:.0f}K'))
axes[0].set_title('Revenue by RFM segment', fontsize=11)
for i, val in enumerate(rfm_plot['revenue']):
    axes[0].text(i, val + 8000, f'${val/1000:.0f}K', ha='center', fontsize=9, color='#444')

# Customer count by segment
axes[1].bar(order, rfm_plot['customers'], color=colors_rfm, alpha=0.8, width=0.55)
axes[1].set_title('Customer count by RFM segment', fontsize=11)
for i, val in enumerate(rfm_plot['customers']):
    axes[1].text(i, val + 2, str(int(val)), ha='center', fontsize=9, color='#444')

plt.suptitle('RFM segmentation — customer value distribution', fontsize=13, y=1.02)
plt.tight_layout()
plt.savefig('fig_rfm.png', dpi=150, bbox_inches='tight')
plt.show()

## 9. Regional analysis

In [ ]:
region = (
    df.groupby('Region').agg(
        revenue  = ('Sales',      'sum'),
        orders   = ('Order ID',   'nunique'),
        customers= ('Customer ID','nunique'),
    )
    .sort_values('revenue', ascending=False)
    .round(2)
)
region['rev_pct'] = (region['revenue'] / region['revenue'].sum() * 100).round(1)

fig, ax = plt.subplots(figsize=(7, 3.5))
colors_reg = [GREEN, BLUE, PURPLE, AMBER]
bars = ax.barh(region.index, region['revenue'], color=colors_reg, height=0.5)
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x/1000:.0f}K'))
ax.set_title('Revenue by region', fontsize=12, pad=10)
ax.invert_yaxis()
for i, (idx, row) in enumerate(region.iterrows()):
    ax.text(row['revenue'] + 4000, i,
            f"${row['revenue']/1000:.0f}K  ({row['rev_pct']}%)",
            va='center', fontsize=9, color='#444')
plt.tight_layout()
plt.savefig('fig_regions.png', dpi=150, bbox_inches='tight')
plt.show()

## 10. Fulfillment — lead time analysis

Lead time (order date → ship date) affects customer satisfaction and logistics cost.
Breaking it down by Ship Mode reveals whether the service tiers are actually differentiated.

In [ ]:
lead = df.groupby('Ship Mode')['lead_days'].agg(['mean', 'median', 'std']).round(2)
print(lead.to_string())

fig, ax = plt.subplots(figsize=(8, 4))
df.boxplot(column='lead_days', by='Ship Mode', ax=ax,
           boxprops=dict(color=BLUE),
           whiskerprops=dict(color=BLUE),
           capprops=dict(color=BLUE),
           medianprops=dict(color=CORAL, linewidth=2),
           flierprops=dict(marker='o', color=GRAY, alpha=0.3, markersize=3))

ax.set_title('Lead time distribution by shipping mode', fontsize=12)
ax.set_xlabel('Ship mode')
ax.set_ylabel('Days from order to shipment')
plt.suptitle('')  # remove matplotlib auto-title
plt.tight_layout()
plt.savefig('fig_leadtime.png', dpi=150, bbox_inches='tight')
plt.show()

## 11. Key findings summary

Consolidating the main takeaways before writing recommendations.

In [ ]:
total_rev     = df['Sales'].sum()
total_orders  = df['Order ID'].nunique()
total_cust    = df['Customer ID'].nunique()
cagr          = ((annual['revenue'].iloc[-1] / annual['revenue'].iloc[0]) ** (1/3) - 1) * 100
champions_rev = rfm_summary.loc['Champions', 'revenue']
champ_share   = champions_rev / total_rev * 100
champ_count   = rfm_summary.loc['Champions', 'customers']

print('=' * 60)
print('EXECUTIVE SUMMARY')
print('=' * 60)
print(f'Total revenue (4Y):     ${total_rev:>12,.0f}')
print(f'Total orders:           {total_orders:>12,}')
print(f'Unique customers:       {total_cust:>12,}')
print(f'Revenue CAGR:           {cagr:>11.1f}%')
print()
print(f'AOV decline 2015→2018:  {annual["aov"].iloc[-1]-annual["aov"].iloc[0]:>+10.0f} USD')
print()
print(f'Champions: {int(champ_count)} customers = {champ_share:.1f}% of revenue')
print()
print('South region gap vs West: '
      f'${(region.loc["West","revenue"]-region.loc["South","revenue"]):,.0f}')
print('=' * 60)

## 12. Recommendations

Based on the analysis, four actionable recommendations:

**1. Address the AOV decline before it compounds**  
Revenue CAGR is strong (15.5%), but AOV fell 12% in four years. Introduce product bundles, cross-sell recommendations at checkout, and minimum-order incentives. Target the Corporate segment (highest AOV at $500).

**2. Protect the Champions cohort**  
197 customers generate 41% of revenue. Build a formal retention program: dedicated account managers for top 50 accounts, quarterly business reviews, early access to new products. Losing 10% of this cohort means ~$100K revenue at risk.

**3. Re-engage the 186 Potential customers**  
They sit just below the Loyal threshold — highest-leverage conversion target. A win-back campaign with personalized offers based on past purchase category can move them up the RFM ladder at low acquisition cost.

**4. Investigate the South region gap**  
South generates $316K less than West despite similar demographic size. Needs a deeper investigation: is it distribution coverage, product-market fit, pricing, or competition? Recommend a targeted cohort analysis and customer survey before investing in expansion.

---
*Analysis by [Your Name] | Tools: Python, pandas, matplotlib, seaborn | Dataset: Superstore 2015–2018*